In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [3]:
df.shape

(2000, 8)

In [4]:
df.head(3)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [5]:
df.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [6]:
df=df.drop('id',axis=1)

In [7]:
answer=df['answer']
df=df.drop('answer',axis=1)

In [8]:
df.shape

(2000, 6)

In [9]:
import re

def clean_text(text):
    text=str(text).lower()
    text=re.sub(r'[^a-zA-Z0-9\s]',' ',text)
    text=re.sub(r'\s+',' ',text).strip()
    return text

df['prompt']=df['prompt'].apply(clean_text)
df['A']=df['A'].apply(clean_text)
df['B']=df['B'].apply(clean_text)
df['C']=df['C'].apply(clean_text)
df['D']=df['D'].apply(clean_text)
df['E']=df['E'].apply(clean_text)

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf=TfidfVectorizer(sublinear_tf=True)
tfidf_train=tfidf.fit_transform(df['prompt']+" "+df['A']+" "+df['B']+" "+df['C']+" "+df['D']+" "+df['E'])

In [11]:
tfidf_train.shape

(2000, 2939)

In [12]:
tfidf_scores = []
for col in ['A', 'B', 'C', 'D', 'E']:
    tfidf_options=tfidf.transform(df[col])
    print(tfidf_options.shape)
    scores = np.diag(cosine_similarity(tfidf_train, tfidf_options))
    tfidf_scores.append(scores)

(2000, 2939)
(2000, 2939)
(2000, 2939)
(2000, 2939)
(2000, 2939)


In [13]:
tfidf_scores=np.array(tfidf_scores).T

In [14]:
tfidf.get_feature_names_out()

array(['000', '013', '013343', ..., 'yukawa', 'zero', 'zweig'],
      shape=(2939,), dtype=object)

In [15]:
from gensim.models import Word2Vec
np.random.seed(1)
from numpy.random import seed
seed(1)

all_text=(df['prompt'].tolist()+ df['A'].tolist()+df['B'].tolist()+df['C'].tolist()+df['D'].tolist()+df['E'].tolist())
sentences=[text.split() for text in all_text]
word2vec=Word2Vec(sentences,vector_size=20,window=1,min_count=1,seed=1,workers=1,epochs=10)

In [16]:
def sentence_embedding(text):
    words=text.split()
    embedding=[word2vec.wv[word] for word in words if word in word2vec.wv]
    if len(embedding)==0:
        return np.zeros(word2vec.vector_size)
    return np.mean(embedding,axis=0)

In [17]:
prompts=df['prompt'].apply(sentence_embedding)
prompts=np.vstack(prompts)

option_cols=['A','B','C','D','E']
w2v_scores=[]

for col in option_cols:
    option_matrix=np.vstack(df[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(prompts,option_matrix))
    w2v_scores.append(scores)
w2v_scores=np.array(w2v_scores).T
print(len(w2v_scores[0]))

5


In [18]:
from transformers import BertTokenizer,BertModel

tokeniser=BertTokenizer.from_pretrained("bert-base-uncased")
model=BertModel.from_pretrained("bert-base-uncased")
model.eval()

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [19]:
import torch 

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)

def bert_embedding(text):
    if not text or not text.strip():
        return np.zeros(768)
    inputs=tokeniser(text,return_tensors='pt',max_length=64,truncation=True)
    inputs={k:v.to(device) for k,v in inputs.items()}
    with torch.no_grad():
        outputs=model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

bert_prompts=np.vstack(df['prompt'].apply(bert_embedding))
option_cols=['A','B','C','D','E']
all_scores=[]

for col in option_cols:
    option_matrix=np.vstack(df[col].apply(bert_embedding))
    scores=np.diag(cosine_similarity(bert_prompts,option_matrix))
    all_scores.append(scores)
all_scores=np.array(all_scores).T
print(len(all_scores[0]))

5


In [20]:
bert_labels=[]
for score in all_scores:
    scored=np.argsort(score)[::-1][:3]
    top_3=[chr(65 + i) for i in scored]
    bert_labels.append(' '.join(top_3))

In [21]:
from sklearn.preprocessing import StandardScaler

ss=StandardScaler()
final1=ss.fit_transform(all_scores)
final2=ss.fit_transform(tfidf_scores)
final3=ss.fit_transform(w2v_scores)

In [22]:
def map_at_3(predictions, ground_truth):
    ap_scores=[1.0 /(pred.index(truth)+1) if truth in pred else 0.0 for pred,truth in zip(predictions, ground_truth)]
    return sum(ap_scores)/len(ap_scores)

In [23]:
def weighted_avg(final1,final2,final3):
    best_score=0
    best_weights=[0.33,0.33,0.33]
    for first in np.arange(0,1.01,0.05):
      for second in np.arange(0,1.01-first,0.05):
        third=1-first-second
        if third<0:
            continue
        final=first*final1+second*final2+third*final3
        labels=[]
        for score in final:
            scored=np.argsort(score)[::-1][:3]
            top_3=[chr(65+i) for i in scored]
            labels.append(top_3)
        score_val=map_at_3(labels,answer)
        if score_val>best_score:
            best_score=score_val
            best_weights=[first,second,third]
    return best_score,best_weights

best_score,best_weights=weighted_avg(final1,final2,final3)

In [24]:
test=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

test['prompt']=test['prompt'].apply(clean_text)
test['A']=test['A'].apply(clean_text)
test['B']=test['B'].apply(clean_text)
test['C']=test['C'].apply(clean_text)
test['D']=test['D'].apply(clean_text)
test['E']=test['E'].apply(clean_text)

test_tfidf=tfidf.transform(test['prompt'])
test_tfidf_scores=[]
for col in option_cols:
    test_tfidf_options=tfidf.transform(test[col])
    scores=np.diag(cosine_similarity(test_tfidf,test_tfidf_options))
    test_tfidf_scores.append(scores)
test_tfidf_scores=np.array(test_tfidf_scores).T

In [25]:
w2v_prompts=np.vstack(test['prompt'].apply(sentence_embedding))
test_w2v_scores=[]
for col in ['A','B','C','D','E']:
    option_matrix=np.vstack(test[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(w2v_prompts,option_matrix))
    test_w2v_scores.append(scores)

test_w2v_scores=np.array(test_w2v_scores).T

In [26]:
test_prompts_bert=np.vstack(test['prompt'].apply(bert_embedding))
test_bert_scores=[]
for col in option_cols:
    option_matrix=np.vstack(test[col].apply(bert_embedding))
    scores=np.diag(cosine_similarity(test_prompts_bert, option_matrix))
    test_bert_scores.append(scores)
test_bert_scores=np.array(test_bert_scores).T

In [27]:
test_final1=ss.fit_transform(test_bert_scores)
test_final2=ss.fit_transform(test_tfidf_scores)
test_final3=ss.fit_transform(test_w2v_scores)

In [28]:
final=best_weights[0]*test_final1+best_weights[1]*test_final2+best_weights[2]*test_final3
test_labels=[]
for score in final:
    scored=np.argsort(score)[::-1][:3]
    top_3=[chr(65+i) for i in scored]
    test_labels.append(" ".join(top_3))

In [29]:
submission=pd.DataFrame({'id':test['id'],'Prediction':test_labels})
submission.to_csv('submission_model1_w2v.csv',index=False)
print("Model submission saved!")
print(submission.head())

Model submission saved!
   id Prediction
0   1      C A E
1   2      E B A
2   3      A D C
3   4      E A C
4   5      E D A
